In [ ]:
# Install required libraries (if not already installed)
!pip install transformers datasets scikit-learn

In [ ]:
# Install sacremoses, a dependency for some tokenizers
!pip install sacremoses

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import os
import json
import random
import shutil
from sklearn.model_selection import train_test_split

# =========================
# PATHS
# =========================
INPUT_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/final_datasets/title_text_gen_ready.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized/biogpt_title_text_gen"
UNSEEN_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/unseen"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(UNSEEN_PATH, exist_ok=True)

# =========================
# SETTINGS
# =========================
RANDOM_SEED = 42
TEST_SIZE = 0.2
MAX_LENGTH = 1024

MODEL_NAME = "microsoft/biogpt"

# =========================
# LOAD DATA
# =========================
data = []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        title = item.get("input", "").strip()
        abstract = item.get("target", "").strip()

        if title and abstract:
            data.append({
                "input": title,
                "target": abstract
            })

print(f"Loaded valid examples: {len(data)}")

# =========================
# SPLIT DATA
# =========================
random.seed(RANDOM_SEED)
random.shuffle(data)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED
)

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")

# Save unseen test set
unseen_file = os.path.join(UNSEEN_PATH, "biogpt_unseen.jsonl")
with open(unseen_file, "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved unseen test set to: {unseen_file}")

# =========================
# TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================
# TOKENIZATION FUNCTION
# =========================
def preprocess(example):
    title = example["input"]
    abstract = example["target"]

    prompt = f"Title: {title}\nAbstract:"
    full_text = f"{prompt} {abstract}{tokenizer.eos_token}"

    tokenized = tokenizer(
        full_text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenized["input_ids"].copy()

    # Ignore padding in loss
    labels = [
        token_id if token_id != tokenizer.pad_token_id else -100
        for token_id in labels
    ]

    tokenized["labels"] = labels
    return tokenized

# =========================
# PROCESS & SAVE
# =========================
def tokenize_and_save(dataset_list, split):
    print(f"\nTokenizing {split} set...")

    ds = Dataset.from_list(dataset_list)

    tokenized = ds.map(preprocess, remove_columns=ds.column_names)

    out_dir = os.path.join(OUTPUT_DIR, split)

    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)

    os.makedirs(out_dir, exist_ok=True)
    tokenized.save_to_disk(out_dir)

    print(f"Saved {split} set to: {out_dir}")
    print(tokenized[0])

tokenize_and_save(train_data, "train")
tokenize_and_save(test_data, "test")

print("\nDone.")